# 03 · Transfer Learning and Fine-Tuning of Persian Transformers

> **Project 3 · Intelligent Analysis of Digikala Reviews**
> Phase goal: fine-tune pretrained encoders on `body` only, to predict **recommendation status** and **rating band** with one shared model.

---

## 🎯 Objective

Fine-tune a pretrained Transformer with **two classification heads** on top of one **shared encoder** (multi-task learning):

| Head | Task | Classes |
| --- | --- | --- |
| `rec` | Recommendation status | `recommended`, `not_recommended`, `no_idea` |
| `band` | Ordered rating band | `low`, `medium`, `high` |

The total loss is the sum of the two head losses. Rows with a missing target are masked out of that head's loss only.

---

## 🧭 Phase Overview

1. **Fine-tune TookaBERT** in two variants:
   - **B**: Step-1 corrected labels, no sample weights.
   - **C**: Step-1 corrected labels plus Step-2 sample weights.
2. **Choose B or C** with a rule fixed before seeing results.
3. **Fine-tune two more encoders** with the winning variant.
4. **Select the primary model** for the semi-supervised phase.
5. **Compare** against the dummy and TF-IDF baselines under the same split and protocol.

---

## 🤖 Encoders

| Model | Tokenizer | `max_length` |
| --- | --- | --- |
| `PartAI/TookaBERT-Base` | BPE (Persian) | 128 |
| `HooshvareLab/roberta-fa-zwnj-base` | Byte-level BPE (Persian) | 128 |
| `microsoft/mdeberta-v3-base` | SentencePiece (multilingual) | 160 |

---

## 📐 Ground Rules

- **Input:** `body` only. No `title`, `rate`, `product_id`, or any other metadata.
- **Labels for training and checkpoint selection:** corrected Step-1 labels only.
- **Data:** full `train_supervised` for training, full validation set for evaluation every epoch.
- **Training:** at most 4 epochs, early stopping with patience 1 on the mean Macro F1 of both heads.
- **Fair comparison:** B and C use identical seeds, batch order, and settings. Only the sample weights differ.
- **No decision uses the test set.** Test metrics are reported on both raw labels (comparable with classmates) and corrected labels.

---

## 🖥️ Where Things Run

| Where | What |
| --- | --- |
| **Training machine** (friend's PC, RTX 4060 Ti, 16 GB) | Fine-tuning and prediction scripts only |
| **This notebook** (local machine) | Analysis of the saved outputs: curves, tables, decisions, charts |

Training is run by `scripts/train_transformer.py` from YAML configs. It resumes automatically from the last saved epoch if interrupted.

---

## 📦 Main Outputs

- Training history, logs, and run summaries for every run
- Validation and test probabilities for both heads
- Saved tokenizer and encoder config (for offline inference)
- Settings table with the reason for each choice
- B vs C decision, encoder comparison, and primary-model selection
- Test metrics, confusion matrices, and four report charts

---

## 🗂️ Notebook Map

| Step | Content |
| --- | --- |
| 1 | Imports and parameters |
| 2 | Load run outputs |
| 3 | Training settings and run comparison tables |
| 4 | Learning curves and overfitting check |
| 5 | B vs C decision |
| 6 | Test metrics and baseline comparison |
| 7 | Confusion matrices |
| 8 | Encoder comparison and primary model |
| 9 | Closeout facts |

---

# Step 0: preparing

In [1]:
# ── Read-only inspection, part 1: project setup, git, ignore rules ──
import json
import subprocess
import tomllib
from pathlib import Path

from digikala_review_transformer import config

root = Path(config.ROOT)


def show(title):
    print(f"\n{'─' * 8} {title} {'─' * 8}")


def run_git(*args):
    result = subprocess.run(["git", "-C", str(root), *args], capture_output=True, text=True)
    return (result.stdout or result.stderr).strip()


# ── 1) pyproject.toml (full text) ──
show("pyproject.toml")
print((root / "pyproject.toml").read_text(encoding="utf-8"))

# ── 1b) Versions locked in uv.lock for the packages we care about ──
show("uv.lock: selected package versions")
lock = tomllib.loads((root / "uv.lock").read_text(encoding="utf-8"))
wanted = [
    "torch", "transformers", "tokenizers", "huggingface-hub", "safetensors",
    "sentencepiece", "protobuf", "accelerate", "pyyaml", "tqdm", "pandas",
    "pyarrow", "scikit-learn", "numpy", "duckdb", "matplotlib",
]
locked = {pkg["name"]: pkg.get("version", "-") for pkg in lock["package"]}
for name in wanted:
    print(f"{name:18s} {locked.get(name, 'NOT IN LOCK')}")

# ── 2) Package __init__.py ──
show("src/digikala_review_transformer/__init__.py")
init_text = (root / "src/digikala_review_transformer/__init__.py").read_text(encoding="utf-8")
print(init_text if init_text.strip() else "(empty file)")

# ── 6) .gitignore and ignore status of the paths this phase will create ──
show(".gitignore")
print((root / ".gitignore").read_text(encoding="utf-8"))

show("git check-ignore on future paths")
sample_paths = [
    "data/training_pack/train.parquet",
    "data/interim/token_cache/tooka_128/tokens.npy",
    "models/pretrained/TookaBERT-Base/model.safetensors",
    "artifacts/transformer_runs/tooka_B/best_model.pt",
    "artifacts/transformer_runs/tooka_B/last.pt",
    "outputs_to_send/tooka_B.zip",
]
for path in sample_paths:
    verdict = run_git("check-ignore", "-v", path)
    print(f"{path}\n    -> {verdict if verdict else 'NOT IGNORED'}")

# ── 7) scripts/ and configs/ ──
show("scripts/ and configs/")
for folder_name in ["scripts", "configs", "src/digikala_review_transformer/training"]:
    folder = root / folder_name
    if folder.exists():
        print(f"{folder_name}/ exists:", sorted(p.name for p in folder.iterdir()))
    else:
        print(f"{folder_name}/ does not exist")
print("tracked in git (scripts, configs):", run_git("ls-files", "scripts", "configs") or "(none)")

# ── 8) Branch and remote (hide any token if the URL contains one) ──
show("git branch and remote")
print("branch:", run_git("branch", "--show-current"))
print(run_git("remote", "-v"))

# ── 9) Notebook numbering ──
show("notebooks/")
for notebook in sorted((root / "notebooks").glob("*.ipynb")):
    print(notebook.name)

# ── 10) Where is a text-normalization function? (source cells only, no outputs) ──
show("'normaliz' in src/*.py")
hits = 0
for py_file in sorted((root / "src").rglob("*.py")):
    for line_number, line in enumerate(py_file.read_text(encoding="utf-8").splitlines(), 1):
        if "normaliz" in line.lower() and hits < 25:
            print(f"{py_file.relative_to(root)}:{line_number}: {line.strip()[:140]}")
            hits += 1
print("matches shown:", hits)

show("'normaliz' in notebook code cells")
hits = 0
for notebook in sorted((root / "notebooks").glob("*.ipynb")):
    cells = json.loads(notebook.read_text(encoding="utf-8")).get("cells", [])
    for cell_index, cell in enumerate(cells):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        matching = [ln.strip() for ln in source.splitlines() if "normaliz" in ln.lower()]
        if matching and hits < 25:
            print(f"{notebook.name} | cell {cell_index}: {matching[0][:140]}")
            hits += 1
print("matches shown:", hits)


──────── pyproject.toml ────────
[project]
name = "digikala-review-transformer"
version = "0.1.0"
description = "Persian Digikala review classification: TF-IDF baseline vs fine-tuned Transformer"
readme = "README.md"
authors = [
    { name = "AliirezzaaZiarati", email = "arzxii1998@gmail.com" }
]
requires-python = ">=3.12"
dependencies = [
    "arabic-reshaper>=3.1.0",
    "duckdb>=2.0.0",
    "huggingface-hub>=2.4.0",
    "joblib>=1.6.0",
    "matplotlib>=3.11.2",
    "nbformat>=5.11.1",
    "pandas>=3.1.1",
    "parsivar>=0.2.3.1",
    "pillow>=13.0.0",
    "plotly>=7.1.0",
    "psutil>=7.2.2",
    "pyarrow>=26.0.1",
    "python-bidi>=0.6.12",
    "scikit-learn>=1.9.1",
    "scipy>=1.18.1",
    "torch>=2.15.0",
    "torchvision>=0.29.2",
    "transformers>=5.22.2",
]

[build-system]
requires = ["uv_build>=0.13.6,<0.14.0"]
build-backend = "uv_build"

[dependency-groups]
dev = [
    "ipykernel>=7.4.0",
]


──────── uv.lock: selected package versions ────────
torch              2.15.0


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [2]:
# ── Read-only inspection, part 1b: normalization search in notebook code cells ──
import json
from pathlib import Path

from digikala_review_transformer import config

root = Path(config.ROOT)

hits = 0
unreadable = []
for notebook in sorted((root / "notebooks").glob("*.ipynb")):
    # Skip notebooks that are empty or not valid JSON, and remember them
    try:
        cells = json.loads(notebook.read_text(encoding="utf-8")).get("cells", [])
    except json.JSONDecodeError:
        unreadable.append((notebook.name, notebook.stat().st_size))
        continue
    for cell_index, cell in enumerate(cells):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        matching = [ln.strip() for ln in source.splitlines() if "normaliz" in ln.lower()]
        if matching and hits < 25:
            print(f"{notebook.name} | cell {cell_index}: {matching[0][:140]}")
            hits += 1

print("matches shown:", hits)
print("unreadable notebooks (name, size in bytes):", unreadable or "none")

00_data_ingest_quality_gate.ipynb | cell 42: "body text normalization; original text stays in raw"
00_data_ingest_quality_gate.ipynb | cell 50: # Total number of rows before normalization
00_data_ingest_quality_gate.ipynb | cell 53: # Replace the old body with the normalized one
00_data_ingest_quality_gate.ipynb | cell 87: fig.text(0.04, 0.865, "Rows whose exact (normalized) review text appears more than once in clean_base",
00_data_ingest_quality_gate.ipynb | cell 91: bg.text(M, 0.17, f"Base: {total:,} reviews, all of clean_base. Words are whitespace-separated tokens of the normalized body. "
01_more_data_ingest_analysis.ipynb | cell 16: # from parsivar import Normalizer
01_more_data_ingest_analysis.ipynb | cell 22: # # Spacing fixes that only the SpellCheck+guard route found, not the Normalizer
01_more_data_ingest_analysis.ipynb | cell 43: print(df["recommendation_status"].value_counts(dropna=False, normalize=True).round(4))
04_label_correction_step1.ipynb | cell 15: # Kneedle core: 

# Inspect: Imports, Labels, Config, Data Columns

In [3]:
# ── Read-only inspection, part 2: imports, labels, config, data columns ──
import ast
import inspect
import subprocess
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

from digikala_review_transformer import config, data_io, labels, utils

root = Path(config.ROOT)
package_dir = root / "src" / "digikala_review_transformer"
processed_dir = Path(config.DATA_PROCESSED)
predictions_dir = root / "reports" / "predictions"


def show(title):
    print(f"\n{'─' * 8} {title} {'─' * 8}")


def run_git(*args):
    result = subprocess.run(["git", "-C", str(root), *args], capture_output=True, text=True)
    return (result.stdout or result.stderr).strip()


# ── A) Which third-party packages does each module import? (light-import check) ──
show("A) imports per module (top-level / inside functions)")
training_ok = {
    "torch", "transformers", "huggingface_hub", "pandas", "pyarrow", "sklearn", "yaml",
    "tqdm", "sentencepiece", "google", "numpy", "tokenizers", "safetensors",
    "digikala_review_transformer",
}


def root_names(nodes):
    names = set()
    for node in nodes:
        if isinstance(node, ast.Import):
            names.update(alias.name.split(".")[0] for alias in node.names)
        elif isinstance(node, ast.ImportFrom) and node.level == 0 and node.module:
            names.add(node.module.split(".")[0])
    return names


for module_name in ["config", "labels", "utils", "data_io", "metrics", "predictions", "plot_style"]:
    tree = ast.parse((package_dir / f"{module_name}.py").read_text(encoding="utf-8"))
    top_level = root_names(tree.body)
    everywhere = root_names(ast.walk(tree))
    nested_only = everywhere - top_level
    outside = sorted(n for n in everywhere if n not in sys.stdlib_module_names and n not in training_ok)
    print(f"{module_name:12s} top={sorted(top_level)} nested_only={sorted(nested_only)}")
    print(f"{'':12s} NOT in training set: {outside or 'none'}")

# ── B) Seed-like constants and simple values in config ──
show("B) config: simple constants")
for name in sorted(vars(config)):
    value = getattr(config, name)
    if name.isupper() and isinstance(value, (int, float, str, bool)):
        print(f"{name} = {value!r}")
path_names = [n for n in vars(config) if n.isupper() and isinstance(getattr(config, n), Path)]
print("number of Path constants:", len(path_names))
print("lines containing 'seed' in config.py:")
for line_number, line in enumerate((package_dir / "config.py").read_text(encoding="utf-8").splitlines(), 1):
    if "seed" in line.lower():
        print(f"  {line_number}: {line.strip()[:140]}")

# ── C) labels: constants and function signatures; data_io.read_split ──
show("C) labels module")
for name in sorted(n for n in vars(labels) if not n.startswith("_")):
    obj = getattr(labels, name)
    if inspect.isfunction(obj) and obj.__module__ == labels.__name__:
        print(f"def {name}{inspect.signature(obj)}")
    elif not (inspect.ismodule(obj) or inspect.isfunction(obj) or inspect.isclass(obj)):
        print(f"{name} = {repr(obj)[:300]}")

show("C2) data_io.read_split and utils.timer")
print("read_split", inspect.signature(data_io.read_split))
print((inspect.getdoc(data_io.read_split) or "(no docstring)")[:400])
print("timer", inspect.signature(utils.timer))

# ── D) Parquet schemas (no data loaded) ──
show("D) parquet schemas")
parquet_paths = [predictions_dir / f"tfidf12_{head}_B__{split}.parquet"
                 for head in ("rec", "band") for split in ("validation", "test")]
parquet_paths.append(processed_dir / "step2_sample_weights.parquet")
for path in parquet_paths:
    parquet_file = pq.ParquetFile(path)
    columns = [f"{field.name}:{field.type}" for field in parquet_file.schema_arrow]
    print(f"{path.name}  rows={parquet_file.metadata.num_rows:,}")
    print("   ", columns)

# ── E) CSV headers and unique label values (label columns only) ──
show("E) CSV headers and unique label values")
for split_name in ["train_supervised", "validation", "test"]:
    csv_path = processed_dir / f"{split_name}.csv"
    header = list(pd.read_csv(csv_path, nrows=0).columns)
    print(f"\n{split_name}.csv columns: {header}")
    label_columns = [c for c in ["recommendation_status", "step1_rec_label",
                                 "rating_band", "step1_rate_band_label"] if c in header]
    label_frame = pd.read_csv(csv_path, usecols=label_columns, dtype=str, keep_default_na=False)
    print(f"  rows: {len(label_frame):,}")
    for column in label_columns:
        print(f"  {column}: {label_frame[column].value_counts(dropna=False).to_dict()}")

# ── F) Git: the empty notebook 06 and tracked files under models/ ──
show("F) git: notebook 06 and models/")
notebook_06 = "notebooks/06_masking_and_artifacts.ipynb"
print("size on disk:", (root / notebook_06).stat().st_size)
print("size in HEAD:", run_git("cat-file", "-s", f"HEAD:{notebook_06}"))
print("last commits touching it:", run_git("log", "--oneline", "-3", "--", notebook_06) or "(none)")
print("git status of it:", run_git("status", "--short", "--", notebook_06) or "(clean)")
print("tracked files under models/:", run_git("ls-files", "models")[:600] or "(none)")


──────── A) imports per module (top-level / inside functions) ────────
config       top=['pathlib'] nested_only=[]
             NOT in training set: none
labels       top=['__future__', 'numpy', 'pandas'] nested_only=[]
             NOT in training set: none
utils        top=['__future__', 'contextlib', 'csv', 'datetime', 'hashlib', 'json', 'pathlib', 'time'] nested_only=[]
             NOT in training set: none
data_io      top=['__future__', 'itertools', 'numpy', 'pandas'] nested_only=[]
             NOT in training set: none
metrics      top=['numpy', 'pandas', 'sklearn'] nested_only=[]
             NOT in training set: none
predictions  top=['datetime', 'numpy', 'pandas', 'pathlib', 'zoneinfo'] nested_only=[]
             NOT in training set: none
plot_style   top=['__future__', 'arabic_reshaper', 'colorsys', 'matplotlib', 'numpy', 'pathlib', 're', 'struct'] nested_only=['bidi']
             NOT in training set: ['arabic_reshaper', 'bidi', 'matplotlib']

──────── B) config: simple

# Inspect: Labels and Metrics Source

In [4]:
# ── Read-only inspection, part 3: labels and metrics source and behavior ──
import inspect

import numpy as np
import pandas as pd

from digikala_review_transformer import labels, metrics


def show(title):
    print(f"\n{'─' * 8} {title} {'─' * 8}")


def print_source(obj_name, module):
    # Print the source of a function, or a note if it does not exist
    obj = getattr(module, obj_name, None)
    show(f"{module.__name__.split('.')[-1]}.{obj_name}")
    print(inspect.getsource(obj) if obj is not None else "NOT FOUND")


print_source("class_weights", labels)
print_source("encode", labels)
print_source("_valid", metrics)
print_source("evaluate", metrics)

show("metrics.per_class_table signature and doc")
print(inspect.signature(metrics.per_class_table))
print(inspect.getdoc(metrics.per_class_table))

# ── Toy calls: one missing label and a prediction that is wrong once ──
show("toy calls")
toy_true = pd.array([0, 1, 2, 1, None], dtype="Int8")
toy_pred = np.array([0, 1, 1, 1, 2])
toy_calls = {
    "class_weights": lambda: labels.class_weights(toy_true),
    "evaluate": lambda: metrics.evaluate(toy_true, toy_pred),
    "per_class_table": lambda: metrics.per_class_table(toy_true, toy_pred, "rec"),
}
for name, call in toy_calls.items():
    try:
        print(f"{name}:\n{call()}\n")
    except Exception as error:
        print(f"{name}: {type(error).__name__}: {error}\n")


──────── labels.class_weights ────────
def class_weights(y, n_classes=3):
    """Balanced weights n / (k * n_c) as {code: weight}, computed from training labels only."""
    y = np.asarray(y, dtype=int)
    counts = np.bincount(y, minlength=n_classes)
    return {i: float(len(y) / (n_classes * c)) for i, c in enumerate(counts)}


──────── labels.encode ────────
def encode(series, head):
    """Map label strings to int codes (nullable Int8). Missing stays <NA>; unknown values raise."""
    out = series.map(label_map(head))
    bad = series.notna() & out.isna()
    if bad.any():
        raise ValueError(f"unknown labels for head '{head}': {series[bad].unique()[:10]}")
    return out.astype("Int8")


──────── metrics._valid ────────
def _valid(y_true, y_pred):
    """Keep only rows with a valid true label; return int arrays."""
    yt = pd.Series(y_true).reset_index(drop=True)
    yp = np.asarray(y_pred)
    assert len(yt) == len(yp), "y_true and y_pred length differ"
    m = yt.notna().

# U2: pyproject.toml، .gitignore و git

In [5]:
# terminal prompts

# Inspect: Windows Wheels in uv.lock and config.py Paths

In [6]:
# ── Read-only inspection, part 4: Windows CUDA wheels in the lock file and config.py ──
import tomllib
from pathlib import Path

from digikala_review_transformer import config

root = Path(config.ROOT)

# ── A) torch / torchvision entries in uv.lock and their Windows cp312 wheels ──
lock = tomllib.loads((root / "uv.lock").read_text(encoding="utf-8"))
for package in lock["package"]:
    if package["name"] in ("torch", "torchvision"):
        wheel_urls = [w.get("url", "") for w in package.get("wheels", [])]
        windows_wheels = [u.rsplit("/", 1)[-1] for u in wheel_urls if "win_amd64" in u and "cp312" in u]
        print(package["name"], package["version"], "| source:", package.get("source"))
        print("   windows cp312 wheels:", windows_wheels or "NONE")

# ── B) How config.py defines paths (first 60 lines) ──
print("\n--- config.py (first 60 lines) ---")
config_path = root / "src" / "digikala_review_transformer" / "config.py"
for line_number, line in enumerate(config_path.read_text(encoding="utf-8").splitlines()[:60], 1):
    print(f"{line_number:3d}: {line}")

torch 2.15.0 | source: {'registry': 'https://pypi.org/simple'}
   windows cp312 wheels: NONE
torch 2.15.0+cu134 | source: {'registry': 'https://download.pytorch.org/whl/cu134'}
   windows cp312 wheels: ['torch-2.15.0%2Bcu134-cp312-cp312-win_amd64.whl']
torchvision 0.29.2 | source: {'registry': 'https://pypi.org/simple'}
   windows cp312 wheels: NONE
torchvision 0.29.2+cu134 | source: {'registry': 'https://download.pytorch.org/whl/cu134'}
   windows cp312 wheels: NONE

--- config.py (first 60 lines) ---
  1: from pathlib import Path
  2: 
  3: # Project root: src/digikala_review_transformer/config.py -> three levels up
  4: ROOT = Path(__file__).resolve().parents[2]
  5: 
  6: # Folders (data and artifacts contents are not tracked by Git)
  7: DATA_RAW = ROOT / "data" / "raw"
  8: DATA_INTERIM = ROOT / "data" / "interim"
  9: DATA_PROCESSED = ROOT / "data" / "processed"
 10: ARTIFACTS = ROOT / "artifacts"
 11: REPORTS = ROOT / "reports"
 12: 
 13: # Cleaned data: DuckDB file lives direc

# Test: Run Config Loader, Hash and Validation Errors

In [7]:
# ── Test: run_config loading, hashing and validation errors ──
import copy
import tempfile
from pathlib import Path

import yaml

from digikala_review_transformer import config
from digikala_review_transformer.training import run_config

configs_dir = Path(config.ROOT) / "configs" / "transformer"
base_path = configs_dir / "base.yaml"
tooka_path = configs_dir / "tooka_B.yaml"
failed = []


def report(name, passed, detail=""):
    print(f"[{'PASS' if passed else 'FAIL'}] {name} {detail}")
    if not passed:
        failed.append(name)


# ── 1) A valid config loads, with the project seed and normalized floats ──
cfg = run_config.load_run_config(tooka_path)
report("tooka_B loads", cfg["run_name"] == "tooka_B" and cfg["training"]["variant"] == "B")
report("seed comes from project config", cfg["seed"] == config.SPLIT_SEED, f"(seed={cfg['seed']})")
report("'extends' is removed", "extends" not in cfg)
report("floats are floats", isinstance(cfg["training"]["lr_encoder"], float))
print("run dir:", run_config.run_dir(cfg))

# ── 2) Hash: deterministic, ignores runtime-only keys, reacts to real changes ──
hash_a = run_config.config_hash(cfg)
report("hash is deterministic", hash_a == run_config.config_hash(run_config.load_run_config(tooka_path))
       and len(hash_a) == 64, f"({hash_a[:12]})")
runtime_changed = copy.deepcopy(cfg)
runtime_changed["training"]["num_workers"] = 0
runtime_changed["training"]["eval_batch_size"] = 64
report("hash ignores runtime-only keys", run_config.config_hash(runtime_changed) == hash_a)
batch_changed = copy.deepcopy(cfg)
batch_changed["training"]["batch_size"] = 16
report("hash changes with batch_size", run_config.config_hash(batch_changed) != hash_a)
variant_changed = copy.deepcopy(cfg)
variant_changed["training"]["variant"] = "C"
report("hash changes with variant", run_config.config_hash(variant_changed) != hash_a)

# ── 3) Resolved config can be written and read back unchanged ──
with tempfile.TemporaryDirectory() as tmp_name:
    tmp = Path(tmp_name)
    run_config.dump_resolved_config(cfg, tmp / "sub" / "config_resolved.yaml")
    reloaded = yaml.safe_load((tmp / "sub" / "config_resolved.yaml").read_text(encoding="utf-8"))
    report("resolved config round-trips", reloaded == cfg)

    # ── 4) Bad configs must be rejected with a clear message ──
    def write_child(name, training=None, top=None):
        training_lines = {"variant": "B"} | (training or {})
        lines = [f"extends: {base_path.as_posix()}", "run_name: t", "model:",
                 "  pretrained_path: p", "  max_length: 128", "training:"]
        lines += [f"  {key}: {value}" for key, value in training_lines.items()]
        lines += [f"{key}: {value}" for key, value in (top or {}).items()]
        path = tmp / name
        path.write_text("\n".join(lines) + "\n", encoding="utf-8")
        return path

    # Control: the helper itself produces a valid config
    try:
        run_config.load_run_config(write_child("control.yaml"))
        report("control child config loads", True)
    except run_config.ConfigError as error:
        report("control child config loads", False, f"\n{error}")

    bad_cases = [
        ("seed in YAML is rejected", {"top": {"seed": 7}}, "seed"),
        ("unknown key is rejected", {"training": {"lr_encodr": "1.0e-5"}}, "lr_encodr"),
        ("encoder LR above head LR is rejected", {"training": {"lr_encoder": "3.0e-4"}}, "lr_encoder"),
        ("unknown variant is rejected", {"training": {"variant": "D"}}, "variant"),
        ("1e-4 read as text is rejected", {"training": {"lr_heads": "1e-4"}}, "lr_heads"),
    ]
    for index, (name, kwargs, snippet) in enumerate(bad_cases):
        path = write_child(f"bad_{index}.yaml", **kwargs)
        try:
            run_config.load_run_config(path)
            report(name, False, "(no error was raised)")
        except run_config.ConfigError as error:
            message = str(error)
            report(name, "[FAIL]" in message and snippet in message)

print("\nALL PASSED" if not failed else f"\nFAILED: {failed}")

[PASS] tooka_B loads 
[PASS] seed comes from project config (seed=2026)
[PASS] 'extends' is removed 
[PASS] floats are floats 
run dir: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/artifacts/transformer_runs/tooka_B
[PASS] hash is deterministic (4d353853f2f5)
[PASS] hash ignores runtime-only keys 
[PASS] hash changes with batch_size 
[PASS] hash changes with variant 
[PASS] resolved config round-trips 
[PASS] control child config loads 
[PASS] seed in YAML is rejected 
[PASS] unknown key is rejected 
[PASS] encoder LR above head LR is rejected 
[PASS] unknown variant is rejected 
[PASS] 1e-4 read as text is rejected 

ALL PASSED


# Inspect: Source Files for the Training Pack

In [8]:
# ---- Section: Imports ----
import inspect
import subprocess
from pathlib import Path

import pyarrow.parquet as pq

from digikala_review_transformer import config, data_io, labels

# ---- Section: Helpers ----
SKIP_DIR_NAMES = {"tfidf_cache", "token_cache", "__pycache__", ".ipynb_checkpoints"}


def list_files(base, suffixes):
    """Return (path, size_mb) for files under base, skipping cache folders."""
    found = []
    if not base.exists():
        return found
    for p in sorted(base.rglob("*")):
        if p.is_file() and p.suffix.lower() in suffixes:
            if SKIP_DIR_NAMES.isdisjoint(p.parts):
                found.append((p, p.stat().st_size / 1e6))
    return found


# ---- Section: CSV and parquet files in data/ ----
print("=== data/ (csv, parquet) ===")
for base in (config.DATA_RAW, config.DATA_INTERIM, config.DATA_PROCESSED):
    for p, mb in list_files(base, {".csv", ".parquet"}):
        print(f"{p.relative_to(config.ROOT)}  [{mb:,.1f} MB]")

# ---- Section: CSV headers of split-like files ----
print("\n=== CSV header lines ===")
for base in (config.DATA_INTERIM, config.DATA_PROCESSED):
    for p, _ in list_files(base, {".csv"}):
        with open(p, encoding="utf-8") as f:
            print(f"{p.relative_to(config.ROOT)}: {f.readline().strip()}")

# ---- Section: Step 2 weight files anywhere in the project ----
print("\n=== Files with 'weight' or 'step2' in the name ===")
for base in (config.DATA_INTERIM, config.DATA_PROCESSED, config.ARTIFACTS, config.REPORTS):
    for p, mb in list_files(base, {".parquet", ".csv", ".json"}):
        low = p.name.lower()
        if "weight" in low or "step2" in low:
            line = f"{p.relative_to(config.ROOT)}  [{mb:,.1f} MB]"
            if p.suffix == ".parquet":
                pf = pq.ParquetFile(p)
                line += f"  rows={pf.metadata.num_rows:,}  cols={pf.schema_arrow.names}"
            print(line)

# ---- Section: Function signatures I will call ----
print("\n=== Signatures ===")
print("data_io.read_split", inspect.signature(data_io.read_split))
print("labels.encode", inspect.signature(labels.encode))
print("labels.class_weights", inspect.signature(labels.class_weights))

# ---- Section: Git commit (for the manifest) ----
print("\n=== Git ===")
try:
    head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=config.ROOT,
                          capture_output=True, text=True, check=True).stdout.strip()
    print("HEAD:", head)
except Exception as exc:
    print("git not usable:", exc)

=== data/ (csv, parquet) ===
data/raw/digikala-comments.csv  [1,278.5 MB]
data/raw/digikala-products.csv  [258.7 MB]
data/interim/body_groups_rating_band.parquet  [0.2 MB]
data/interim/body_groups_recommendation_status.parquet  [0.2 MB]
data/interim/cap_dropped_rows.parquet  [6.1 MB]
data/interim/cap_overcap_bodies.parquet  [0.0 MB]
data/interim/dropped_tables_archive/body_backup_latin.parquet  [0.1 MB]
data/interim/dropped_tables_archive/defect_manifest.parquet  [10.3 MB]
data/interim/dropped_tables_archive/dup_plan.parquet  [0.1 MB]
data/interim/dropped_tables_archive/dup_plan_s3.parquet  [0.0 MB]
data/interim/dropped_tables_archive/len_sample.parquet  [17.7 MB]
data/interim/dropped_tables_archive/rule_summary.parquet  [0.0 MB]
data/interim/explorer_candidates_tfidf12/tfidf12_band_B.parquet  [0.8 MB]
data/interim/explorer_candidates_tfidf12/tfidf12_rec_B.parquet  [0.9 MB]
data/interim/step2_oof/oof_band_fold0.parquet  [12.9 MB]
data/interim/step2_oof/oof_band_fold1.parquet  [12.9 MB]